# Parsing, tokenization & phrase detection

_Feature Engineering_

**Keep meaningful multi-word phrases like "New York" as single features instead of splitting them into useless single words.**

Raw text is just a string of characters. A model needs numbers, so the first job is to chop the string into tokens (usually words).

       The naive way treats each word as its own feature. This is the bag-of-words (BoW) model. It throws away word order and, worse, it splits real phrases.

---

This notebook is a practice scaffold for the **Parsing, tokenization & phrase detection** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
!pip install -q river
import numpy as np, matplotlib.pyplot as plt

## See it for yourself: the problem, then the fix

Run this cell. It plots the **raw** data and reproduces the problem it causes, then plots the **engineered** data and shows the fix — with before/after numbers.

In [ ]:
# Phrase / collocation detection: raw bigram frequency vs PMI
# LESSON: real multi-word phrases ("machine learning", "new york") are found
# by a STATISTICAL score (pointwise mutual information, PMI), not by counting.
# PROBLEM: ranking word pairs by RAW FREQUENCY surfaces filler ("of the",
# "in the") because common words co-occur a lot just by being common.
# FIX: PMI = log( p(a,b) / (p(a) p(b)) ) divides out how common each word is,
# so the true phrase rises to the top.

import numpy as np
from collections import Counter
import matplotlib.pyplot as plt

# ----------------------------------------------------------------------
# 1. LOAD REAL DATA: a small inline corpus (no downloads).
#    The true phrase "machine learning" recurs amid lots of filler words
#    ("the", "of", "in", "a"). "new york" also appears a couple of times.
# ----------------------------------------------------------------------
corpus = """
in the study of the field of the data the machine learning idea is in the air .
in the city of the team in the office in the building works on the data .
of the goal of the work of the day the team in the room reads of the news .
in the spring in the city of new york in the heart of the town it is nice .
of the data of the model of the team of the lab the work is in the report .
in the field of the work in the area of the task the machine learning helps .
of the city of new york of the place of the visit the trip is in the plan .
machine learning is in the news of the day of the year of the era of the world .
""".lower().split()

print(f"corpus length: {len(corpus)} tokens")

# Build unigram and bigram counts with collections.Counter
unigrams = Counter(corpus)
bigrams = Counter(zip(corpus[:-1], corpus[1:]))
N_uni = sum(unigrams.values())
N_bi = sum(bigrams.values())

# ----------------------------------------------------------------------
# 2. VISUALIZE THE PURE (raw) DATA: how often each word occurs.
#    The filler words ("the", "of", "in") dominate the raw word counts —
#    this is exactly why they will also dominate raw bigram counts.
# ----------------------------------------------------------------------
top_words = unigrams.most_common(8)
plt.figure(figsize=(7, 3))
plt.bar([w for w, _ in top_words], [c for _, c in top_words], color="#888")
plt.title("Raw word counts: filler words dominate the corpus")
plt.ylabel("count")
plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------
# 3. REPRODUCE THE PROBLEM on raw data: rank bigrams by RAW FREQUENCY.
#    The top pairs are meaningless common-word pairs, NOT real phrases.
# ----------------------------------------------------------------------
raw_ranking = bigrams.most_common(8)
print("\nTop bigrams by RAW FREQUENCY (the PROBLEM — filler at the top):")
for (a, b), c in raw_ranking:
    print(f"  {a} {b:<10} count={c}")

# ----------------------------------------------------------------------
# 4. APPLY THE TECHNIQUE: score each bigram by PMI.
#    PMI(a,b) = log( p(a,b) / (p(a) * p(b)) )
#      p(a,b) = count(a,b) / (#bigrams)          -> how often they co-occur
#      p(a), p(b) = count / (#unigrams)          -> how common each word is
#    Dividing by p(a)*p(b) cancels out raw commonness, so glue-words score low.
#    Require count >= 2 so we score recurring pairs, not one-off flukes.
# ----------------------------------------------------------------------
pmi_scores = {}
for (a, b), c_ab in bigrams.items():
    if c_ab < 2:
        continue
    p_ab = c_ab / N_bi
    p_a = unigrams[a] / N_uni
    p_b = unigrams[b] / N_uni
    pmi_scores[(a, b)] = np.log(p_ab / (p_a * p_b))

pmi_ranking = sorted(pmi_scores.items(), key=lambda kv: kv[1], reverse=True)[:8]

# Visualize the ENGINEERED scores: PMI per bigram, true phrase on top.
plt.figure(figsize=(7, 3))
labels = [f"{a} {b}" for (a, b), _ in pmi_ranking]
plt.barh(labels[::-1], [s for _, s in pmi_ranking][::-1], color="#2a7")
plt.title("PMI per bigram: real phrases rise to the top")
plt.xlabel("PMI (higher = stronger phrase)")
plt.tight_layout()
plt.show()

print("\nTop bigrams by PMI (the FIX — real phrases at the top):")
for (a, b), s in pmi_ranking:
    print(f"  {a} {b:<10} PMI={s:.3f}")

# ----------------------------------------------------------------------
# 5. SHOW THE FIX side by side: raw-frequency ranking vs PMI ranking.
#    The real phrase "machine learning" tops the PMI list but is buried
#    (or absent) at the top of the raw-frequency list.
# ----------------------------------------------------------------------
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

raw_labels = [f"{a} {b}" for (a, b), _ in raw_ranking]
raw_counts = [c for _, c in raw_ranking]
ax[0].barh(raw_labels[::-1], raw_counts[::-1], color="#888")
ax[0].set_title("PROBLEM: top bigrams by RAW FREQUENCY\n(filler wins)")
ax[0].set_xlabel("count")

pmi_labels = [f"{a} {b}" for (a, b), _ in pmi_ranking]
pmi_vals = [s for _, s in pmi_ranking]
colors = ["#2a7" if lab in ("machine learning", "new york") else "#888"
          for lab in pmi_labels]
ax[1].barh(pmi_labels[::-1], pmi_vals[::-1], color=colors[::-1])
ax[1].set_title("FIX: top bigrams by PMI\n(real phrases win)")
ax[1].set_xlabel("PMI")

plt.tight_layout()
plt.show()

# ----------------------------------------------------------------------
# One-line before/after summary: where does "machine learning" rank?
# ----------------------------------------------------------------------
raw_order = [f"{a} {b}" for (a, b), _ in bigrams.most_common()]
pmi_order = [f"{a} {b}" for (a, b), _ in
             sorted(pmi_scores.items(), key=lambda kv: kv[1], reverse=True)]
raw_rank = raw_order.index("machine learning") + 1
pmi_rank = pmi_order.index("machine learning") + 1
print(f"\n'machine learning' rank  PROBLEM (raw freq): #{raw_rank}"
      f"   →   FIX (PMI): #{pmi_rank}")

## Reference implementation — nltk / spaCy / pandas

In [ ]:
import json
import pandas as pd
import spacy
import nltk
from nltk.collocations import BigramAssocMeasures, BigramCollocationFinder

# --- Load a slice of the Yelp reviews dataset (from the book's GitHub) ---
# github.com/alicezheng/feature-engineering-book
f = open('data/yelp/v6/yelp_academic_dataset_review.json')
js = [json.loads(f.readline()) for _ in range(10000)]
f.close()
review_df = pd.DataFrame(js)

# --- 1) Tokenization & parsing with spaCy (also gives noun chunks) ---
nlp = spacy.load('en_core_web_sm')
doc_df = review_df['text'].apply(nlp)          # parse each review

# spaCy tokenizes AND tags part-of-speech in one pass
for tok in doc_df[0]:
    print(tok.text, tok.pos_)                  # token + POS tag

# spaCy noun-chunk parsing: grammar-based noun phrases ("happy hour", "wait staff")
print([chunk.text for chunk in doc_df[0].noun_chunks])

# --- 2) Statistical phrase detection with nltk collocations ---
# Flatten all reviews into one token stream (lowercased words)
nltk.download('punkt')
words = [w.lower() for review in review_df['text']
                   for w in nltk.word_tokenize(review)
                   if w.isalpha()]

bigram_measures = BigramAssocMeasures()
finder = BigramCollocationFinder.from_words(words)
finder.apply_freq_filter(10)                   # frequency floor: drop rare bigrams

# Likelihood-ratio test: the book's preferred collocation score
print("Top by likelihood ratio:")
print(finder.nbest(bigram_measures.likelihood_ratio, 20))

# Pointwise mutual information: the same idea, simpler formula
print("Top by PMI:")
print(finder.nbest(bigram_measures.pmi, 20))

# Contrast: raw frequency over-ranks filler like ('of', 'the')
print("Top by raw frequency (note the junk):")
print(finder.nbest(bigram_measures.raw_freq, 20))

## Visualize the data & results

_On a small real corpus where "machine learning" repeats, does pointwise mutual information (PMI) score the true phrase far above incidental word pairs?_

In [ ]:
import numpy as np

# A small REAL corpus: the phrase "machine learning" repeats on purpose.
corpus = (
    "machine learning is fun . machine learning powers a model in a lab . "
    "deep learning extends machine learning . a model uses deep learning . "
    "in a lab we study machine learning and we study deep learning . "
    "the cost of a model and the price of a dataset matter in the end . "
    "the team in an office near a river by a lake in a big city of the world . "
    "many of the books on the shelves of a room of the house in a quiet town . "
    "we love machine learning . they teach deep learning . the model in a room . "
    "of course the of the in the the of of in the a the of in in the of the ."
)

tokens = corpus.split()
N = len(tokens)
bigrams = list(zip(tokens[:-1], tokens[1:]))
B = len(bigrams)

# Unigram and bigram counts (numpy only, dict-based counting)
def counts(seq):
    keys, c = np.unique(np.array(seq, dtype=object), return_counts=True)
    return dict(zip(keys.tolist(), c.tolist()))

uni = counts(tokens)
bi  = counts([a + " " + b for a, b in bigrams])

def pmi(a, b):
    p_a  = uni[a] / N
    p_b  = uni[b] / N
    p_ab = bi[a + " " + b] / B
    return float(np.log(p_ab / (p_a * p_b)))

candidates = [("machine", "learning"), ("deep", "learning"),
              ("of", "the"), ("in", "the"), ("the", "model")]
for a, b in candidates:
    print(f"{a} {b}: PMI = {pmi(a, b):.2f}")

# -> machine learning: PMI = 2.67
#    deep learning:    PMI = 2.67
#    of the:           PMI = 1.16
#    in the:           PMI = 1.02
#    the model:        PMI = 0.65

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** In a 1000-word corpus, "happy" appears 20 times, "hour" appears 25 times, and the bigram "happy hour" appears 15 times (out of 999 bigrams). Is this a phrase? Compute $\mathrm{PMI}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Word probabilities: $p(\text{happy}) = 20/1000 = 0.02$, $p(\text{hour}) = 25/1000 = 0.025$. — _Each word probability is its count over the total word count._
- Joint probability: $p(a,b) = 15/999 \approx 0.015$. — _The bigram probability is the pair count over the number of bigrams._
- Chance baseline: $p(a)\,p(b) = 0.02 \times 0.025 = 0.0005$. — _Independent words would co-occur this often, given how common each is._
- Ratio: $0.015 / 0.0005 = 30$. So $\mathrm{PMI} = \log(30) \approx 3.40$. — _Co-occurs 30 times more than chance; a large positive PMI._

**Answer:** Yes, a strong phrase: ratio 30, $\mathrm{PMI} \approx 3.40$.

</details>

**Problem 2.** "of the" is the most frequent bigram in a corpus. Why is raw frequency a bad way to decide it is a phrase, and what fixes it?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note that "of" and "the" are each extremely common on their own. — _Common words land next to each other often, with no special meaning._
- Raw frequency counts only $p(a,b)$ and ignores $p(a)$ and $p(b)$. — _It cannot tell "frequent because meaningful" from "frequent because both words are everywhere"._
- PMI divides by the chance baseline $p(a)\,p(b)$, which is huge for "of the", so the ratio collapses toward 1. — _Dividing by the baseline cancels the effect of commonness._

**Answer:** Raw frequency over-ranks filler. A statistical score (PMI or likelihood ratio) divides out commonness, so "of the" scores near zero.

</details>

**Problem 3.** A bigram appears exactly once, and both its words are rare. Its PMI is enormous. Should you keep it as a phrase? What guards against this?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall PMI: rare words make $p(a)\,p(b)$ tiny, so a single co-occurrence gives a giant ratio. — _PMI is unstable when counts are tiny; one accident looks like strong attraction._
- One observation is not evidence of a reliable phrase. — _It could easily be a coincidence; nothing repeats it._
- Apply a frequency floor: drop bigrams seen fewer than, say, 5 or 10 times before scoring. — _Only pairs with enough evidence survive, so PMI's instability is contained._

**Answer:** No — discard it. A frequency floor before scoring guards against PMI's rare-word instability.

</details>